In [6]:
# ============================================================
# INSTALL REQUIRED LIBRARY
# ============================================================

# Install pmdarima for automatic ARIMA model selection
!pip install pmdarima

In [7]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import pandas as pd
import numpy as np

from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error

In [8]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

# Required when running the notebook in Google Colab
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# ============================================================
# DEFINE FILE PATHS
# ============================================================

# Check whether the notebook is running with the dataset
# available in the specified Google Drive location

if os.path.exists(
    "/content/drive/MyDrive/research/Nifty 50 Historical Data (2014-2024) Daily.csv"
):

    # Paths for Google Colab
    historical_path = (
        "/content/drive/MyDrive/research/"
        "Nifty 50 Historical Data (2014-2024) Daily.csv"
    )

    future_path = (
        "/content/drive/MyDrive/research/"
        "Nifty 50 Historical Data (2024-25).csv"
    )

else:

    # Paths for GitHub / local execution
    historical_path = (
        "Nifty 50 Historical Data (2014-2024) Daily.csv"
    )

    future_path = (
        "Nifty 50 Historical Data (2024-25).csv"
    )


# Verify whether the files are available
print("Historical file exists:", os.path.exists(historical_path))
print("Future file exists:", os.path.exists(future_path))

Historical file exists: True
Future file exists: True


In [10]:
# ============================================================
# LOAD HISTORICAL DATA
# ============================================================

# Load historical NIFTY 50 daily data
hist_df = pd.read_csv(historical_path)

# Display the first five observations
hist_df.head()

,Date,Price,Open,High,Low,Vol.
0,27-11-2024,24278.20,24235.40,24353.60,24146.60,295010000.0
1,26-11-2024,24194.50,24343.30,24343.30,24125.40,230690000.0
2,25-11-2024,24221.90,24253.55,24351.55,24135.45,687170000.0
3,22-11-2024,23907.25,23411.80,23956.10,23359.00,367560000.0
4,21-11-2024,23349.90,23488.45,23507.30,23263.15,420330000.0


In [11]:
# ============================================================
# DATA CLEANING AND PREPARATION
# ============================================================

# Convert price-related columns from string format
# to numerical format by removing commas

for col in ["Price", "High", "Low"]:

    hist_df[col] = (
        hist_df[col]
        .astype(str)
        .str.replace(",", "")
        .astype(float)
    )


# Convert Date column into datetime format
hist_df["Date"] = pd.to_datetime(
    hist_df["Date"],
    format="%d-%m-%Y"
)


# Sort the dataset chronologically
hist_df = hist_df.sort_values("Date")


# Display dataset information
print("Historical Dataset Shape:", hist_df.shape)

hist_df.head()

Historical Dataset Shape: (2698, 6)


,Date,Price,Open,High,Low,Vol.
2697,2014-01-01,6301.65,6323.80,6327.2,6298.25,69570000.0
2696,2014-01-02,6221.15,6301.25,6358.3,6211.30,158130000.0
2695,2014-01-03,6211.15,6194.55,6221.7,6171.25,139040000.0
2694,2014-01-06,6191.45,6220.85,6224.7,6170.25,118350000.0
2693,2014-01-07,6162.25,6203.90,6221.5,6144.75,138560000.0


In [12]:
# ============================================================
# CREATE TIME SERIES
# ============================================================

# Set Date as the index and select the Price column
# Price represents the closing price

ts = hist_df.set_index("Date")["Price"]

# Display the time series
ts.head()

,Price
Date,
2014-01-01,6301.65
2014-01-02,6221.15
2014-01-03,6211.15
2014-01-06,6191.45
2014-01-07,6162.25


In [13]:
# ============================================================
# ARIMA MODEL SELECTION
# ============================================================

# Automatically identify the optimal ARIMA model
# based on the historical NIFTY 50 closing price series

model = auto_arima(
    ts,

    seasonal=False,       # No seasonal component
    stepwise=True,        # Faster model selection
    suppress_warnings=True
)


# Display the selected ARIMA model
print(model.summary())

                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 2698
Model:               SARIMAX(0, 1, 0)   Log Likelihood              -16948.332
Date:                Sat, 05 Sep 2026   AIC                          33900.664
Time:                        10:40:18   BIC                          33912.463
Sample:                             0   HQIC                         33904.931
                               - 2698                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      6.6654      2.589      2.575      0.010       1.592      11.739
sigma2      1.682e+04    181.963     92.444      0.000    1.65e+04    1.72e+04
Ljung-Box (L1) (Q):                   0.57   Jarque-

In [14]:
# ============================================================
# LOAD FUTURE DATA
# ============================================================

# Load future NIFTY 50 data
future_df = pd.read_csv(future_path)


# Convert Date column into datetime format
future_df["Date"] = pd.to_datetime(
    future_df["Date"]
)


# Sort the dataset chronologically
future_df = future_df.sort_values("Date")


# Convert Price column into numeric format
future_df["Price"] = (
    future_df["Price"]
    .astype(str)
    .str.replace(",", "")
    .astype(float)
)


# Display the first five observations
future_df.head()

/tmp/ipykernel_759/3786557809.py:10: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  future_df["Date"] = pd.to_datetime(


,Date,Price,Open,High,Low,Vol.,Change %
0,2024-11-28,23914.15,"24,274.15","24,345.75","23,873.35",366.75M,-1.49%
1,2024-11-29,24131.10,"23,927.15","24,188.45","23,927.15",282.10M,0.91%
2,2024-12-02,24276.05,"24,140.85","24,301.70","24,008.65",220.38M,0.60%
3,2024-12-03,24457.15,"24,367.50","24,481.35","24,280.00",339.47M,0.75%
4,2024-12-04,24467.45,"24,488.75","24,573.20","24,366.30",348.00M,0.04%


In [15]:
# ============================================================
# SELECT FIRST 90 DAYS
# ============================================================

# Select the first 90 observations from the future dataset
# These values will be used for model evaluation

future_90 = future_df.head(90)


# Extract the corresponding dates
future_dates = future_90["Date"]


# Check the selected data
print("Number of observations:", len(future_90))

future_90.head()

Number of observations: 90


,Date,Price,Open,High,Low,Vol.,Change %
0,2024-11-28,23914.15,"24,274.15","24,345.75","23,873.35",366.75M,-1.49%
1,2024-11-29,24131.10,"23,927.15","24,188.45","23,927.15",282.10M,0.91%
2,2024-12-02,24276.05,"24,140.85","24,301.70","24,008.65",220.38M,0.60%
3,2024-12-03,24457.15,"24,367.50","24,481.35","24,280.00",339.47M,0.75%
4,2024-12-04,24467.45,"24,488.75","24,573.20","24,366.30",348.00M,0.04%


In [16]:
# ============================================================
# GENERATE 90-DAY FORECAST
# ============================================================

# Generate forecasts for the next 90 periods
forecast = model.predict(
    n_periods=90
)


# Create a dataframe containing
# forecast dates and predicted closing prices

forecast_df = pd.DataFrame({

    "Date": future_dates.values,

    "Predicted_Close": forecast
})


# Display forecasted values
forecast_df.head()

/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


,Date,Predicted_Close
2698,2024-11-28,24284.865387
2699,2024-11-29,24291.530775
2700,2024-12-02,24298.196162
2701,2024-12-03,24304.861550
2702,2024-12-04,24311.526937


In [17]:
# ============================================================
# ACTUAL VS PREDICTED COMPARISON
# ============================================================

# Merge actual closing prices with predicted closing prices

comparison_df = pd.merge(

    future_90[["Date", "Price"]],

    forecast_df,

    on="Date",

    how="inner"
)


# Rename the actual price column
comparison_df.rename(
    columns={
        "Price": "Actual_Close"
    },

    inplace=True
)


# Display comparison dataframe
comparison_df.head()

,Date,Actual_Close,Predicted_Close
0,2024-11-28,23914.15,24284.865387
1,2024-11-29,24131.10,24291.530775
2,2024-12-02,24276.05,24298.196162
3,2024-12-03,24457.15,24304.861550
4,2024-12-04,24467.45,24311.526937


In [18]:
# ============================================================
# MODEL PERFORMANCE EVALUATION
# ============================================================

# Calculate Mean Squared Error (MSE)

mse = mean_squared_error(

    comparison_df["Actual_Close"],

    comparison_df["Predicted_Close"]
)


# Calculate Root Mean Squared Error (RMSE)

rmse = np.sqrt(mse)


# Calculate Mean Absolute Percentage Error (MAPE)

mape = np.mean(

    np.abs(

        (
            comparison_df["Actual_Close"]
            -
            comparison_df["Predicted_Close"]
        )

        /

        comparison_df["Actual_Close"]
    )

) * 100


# Display evaluation metrics

print("Model Performance Metrics")

print("-------------------------")

print("MSE  :", mse)

print("RMSE :", rmse)

print("MAPE :", mape, "%")

Model Performance Metrics
-------------------------
MSE  : 2122720.2792840353
RMSE : 1456.9558261265286
MAPE : 5.526953494385257 %


In [19]:
# ============================================================
# FINAL RESULTS
# ============================================================

print("Selected ARIMA Model:")

print(model.order)

print("\nForecast Period:")

print("90 Days")

print("\nEvaluation Metrics:")

print("MSE  :", mse)

print("RMSE :", rmse)

print("MAPE :", mape, "%")

Selected ARIMA Model:
(0, 1, 0)

Forecast Period:
90 Days

Evaluation Metrics:
MSE  : 2122720.2792840353
RMSE : 1456.9558261265286
MAPE : 5.526953494385257 %
